# QUELL - Step 04: Klasik ML baseline (XGBoost + Random Forest) — v2

Duzeltmeler: metrikler class-ismi uzerinden (train'de eksik class olsa da patlamaz, 0 F1 alir), zaman kolonu (frame.time) featurelerden dusuruldu, train'de eksik class uyarisi. Tek hucreyi Shift+Enter; sonundaki BASELINE OZET'i paylas.

In [ ]:
import os, json, glob, sys, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
try: import xgboost as xgb
except Exception:
    subprocess.run([sys.executable,"-m","pip","install","-q","xgboost"]); import xgboost as xgb

ROOT=Path.home()/"quell-edge-llm-ids"; PROC=ROOT/"data"/"processed"; SPL=ROOT/"splits"; RES=ROOT/"results"
SEED=42
rep=json.load(open(RES/"split_report.json"))
LABELISH={"label","attack","attack_type","attack_label","type","class","category","marker","__label__"}
BASE={}

def prep(name):
    meta=rep[name]; label=meta["label_col"]; group=meta.get("group_col"); tcol=meta.get("time_col")
    df=pd.read_parquet(PROC/f"{name}.parquet")
    sp=np.load(SPL/f"{name}_split.npz"); tr,te=sp["train"],sp["test"]
    drop=set([label]); drop|=set(meta.get("leaky_candidates",[]))
    if group: drop.add(group)
    if tcol: drop.add(tcol)                        # the time column is NOT a feature (leakage)
    for c in df.columns:
        if c!=label and c.lower() in LABELISH: drop.add(c)   # secondary labels
    feats=[c for c in df.columns if c not in drop]
    X=df[feats].copy()
    obj=[c for c in feats if not is_numeric_dtype(X[c])]
    if obj:
        oe=OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1)
        oe.fit(X.iloc[tr][obj].astype(str)); X[obj]=oe.transform(X[obj].astype(str))
    for c in feats:
        if not is_numeric_dtype(X[c]): X[c]=pd.to_numeric(X[c],errors="coerce")
    X=X.fillna(X.iloc[tr].median(numeric_only=True)).fillna(-1)
    ys=df[label].astype(str)
    classes_all=sorted(ys.unique())
    return X.iloc[tr],X.iloc[te],ys.iloc[tr].reset_index(drop=True),ys.iloc[te].reset_index(drop=True),classes_all,feats,label

def evaluate(name):
    print(f"\n########## {name} ##########",flush=True)
    Xtr,Xte,ytr_s,yte_s,classes_all,feats,label=prep(name)
    miss=sorted(set(classes_all)-set(pd.unique(ytr_s)))
    print(f"feature:{len(feats)} | class:{len(classes_all)} | train:{len(Xtr):,} test:{len(Xte):,} | label:{label}",flush=True)
    if miss: print("  WARNING: classes not present at all in train (not learned in this split):",miss,flush=True)
    le=LabelEncoder().fit(ytr_s); ytr=le.transform(ytr_s)
    models={
      "xgboost":xgb.XGBClassifier(tree_method="hist",n_estimators=300,max_depth=8,learning_rate=0.3,
                 subsample=0.9,colsample_bytree=0.9,n_jobs=-1,eval_metric="mlogloss",random_state=SEED),
      "random_forest":RandomForestClassifier(n_estimators=100,n_jobs=-1,random_state=SEED),
    }
    BASE[name]={"n_features":len(feats),"n_classes":len(classes_all),"classes":classes_all,
                "classes_missing_in_train":miss,"n_train":int(len(Xtr)),"n_test":int(len(Xte)),"models":{}}
    for mname,model in models.items():
        t=time.time(); model.fit(Xtr,ytr); pred=le.inverse_transform(model.predict(Xte)); dt=time.time()-t
        acc=accuracy_score(yte_s,pred)
        mf1=f1_score(yte_s,pred,average="macro",labels=classes_all,zero_division=0)
        wf1=f1_score(yte_s,pred,average="weighted",labels=classes_all,zero_division=0)
        rp=classification_report(yte_s,pred,labels=classes_all,target_names=classes_all,output_dict=True,zero_division=0)
        pc={k:round(rp[k]["f1-score"],4) for k in classes_all}
        BASE[name]["models"][mname]={"accuracy":round(float(acc),4),"macro_f1":round(float(mf1),4),
                                     "weighted_f1":round(float(wf1),4),"fit_predict_sec":round(dt,1),"per_class_f1":pc}
        print(f"  {mname:14s} acc={acc:.4f}  macroF1={mf1:.4f}  weightedF1={wf1:.4f}  ({dt:.0f}s)",flush=True)

for nm in ["edge_iiotset","ciciot2023","nbaiot"]:
    try: evaluate(nm)
    except Exception as e:
        import traceback; print(f"{nm} HATA:",e); traceback.print_exc()

json.dump(BASE,open(RES/"baseline_report.json","w"),indent=2,ensure_ascii=False)
print("\n=================  BASELINE OZET (macro-F1)  =================",flush=True)
print(f"{'dataset':16s}{'XGBoost':>12s}{'RandomForest':>16s}")
for nm,v in BASE.items():
    x=v["models"].get("xgboost",{}).get("macro_f1","-"); r=v["models"].get("random_forest",{}).get("macro_f1","-")
    print(f"{nm:16s}{str(x):>12s}{str(r):>16s}")
print("\nsaved -> results/baseline_report.json")
